In [1]:
import logging

import numpy as np
import pandas as pd
import akshare as ak

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [2]:
df = ak.stock_zh_a_tick_tx_js(symbol='sz002487')

In [14]:
df.to_excel('1.xlsx')

In [3]:
df.head()

,成交时间,成交价格,价格变动,成交量,成交金额,性质
0,09:25:00,70.36,0.46,1315,9252340,买盘
1,09:30:00,70.50,0.14,834,5871373,买盘
2,09:30:03,70.48,-0.02,374,2638263,卖盘
3,09:30:06,70.42,-0.06,652,4590780,卖盘
4,09:30:09,70.37,-0.05,264,1858127,卖盘


In [22]:
# ===== 1️⃣ 转时间格式 =====
df["成交时间"] = pd.to_datetime(df["成交时间"])

# ===== 2️⃣ 提取分钟 =====
df["分钟"] = df["成交时间"].dt.floor("min")

# ===== 3️⃣ 标记买卖 =====
df["买量"] = df.apply(lambda x: x["成交量"] if x["性质"] == "买盘" or (x["性质"]=="中性盘" and x["价格变动"]>0) else 0, axis=1)

df["卖量"] = df.apply(lambda x: x["成交量"] if x["性质"] == "卖盘" or (x["性质"]=="中性盘" and x["价格变动"]<0) else 0, axis=1)

# ===== 4️⃣ 按分钟聚合 =====
minute_df = df.groupby("分钟").agg({
    "买量": "sum",
    "卖量": "sum",
    "成交量": "sum"
}).reset_index()

# ===== 5️⃣ 计算买卖比 =====
minute_df["买卖比"] = minute_df["买量"] / (minute_df["卖量"] + 1e-6)
minute_df["买卖差"] = minute_df["买量"] - minute_df["卖量"]
minute_df["买卖比"] = minute_df["买卖比"].round(2)
minute_df["买卖比"] = minute_df["买卖比"].map(lambda x: f"{x:.2f}")
print(minute_df)

                     分钟    买量    卖量   成交量            买卖比   买卖差
0   2026-03-22 09:25:00  1315     0  1315  1315000000.00  1315
1   2026-03-22 09:30:00  3166  3218  6569           0.98   -52
2   2026-03-22 09:31:00  2424   774  3198           3.13  1650
3   2026-03-22 09:32:00  1556  1435  2991           1.08   121
4   2026-03-22 09:33:00  1069  1728  2797           0.62  -659
..                  ...   ...   ...   ...            ...   ...
236 2026-03-22 14:54:00   388  1050  1438           0.37  -662
237 2026-03-22 14:55:00  1304   916  2220           1.42   388
238 2026-03-22 14:56:00  1184  1671  2855           0.71  -487
239 2026-03-22 14:57:00    58     0    58    58000000.00    58
240 2026-03-22 15:00:00     0  3583  3583           0.00 -3583

[241 rows x 6 columns]


In [23]:
minute_df["order_flow"] = (
    minute_df["买量"] - minute_df["卖量"]
) / (minute_df["成交量"] + 1e-6)

In [24]:
minute_df["买占比"] = minute_df["买量"] / (minute_df["成交量"] + 1e-6)

In [25]:
minute_df.head(20)

,分钟,买量,卖量,成交量,买卖比,买卖差,order_flow,买占比
0,2026-03-22 09:25:00,1315,0,1315,1315000000.00,1315,1.000000,1.000000
1,2026-03-22 09:30:00,3166,3218,6569,0.98,-52,-0.007916,0.481961
2,2026-03-22 09:31:00,2424,774,3198,3.13,1650,0.515947,0.757974
3,2026-03-22 09:32:00,1556,1435,2991,1.08,121,0.040455,0.520227
4,2026-03-22 09:33:00,1069,1728,2797,0.62,-659,-0.235610,0.382195
5,2026-03-22 09:34:00,323,2004,2327,0.16,-1681,-0.722389,0.138805
6,2026-03-22 09:35:00,1579,1169,2774,1.35,410,0.147801,0.569214
7,2026-03-22 09:36:00,2207,654,2861,3.37,1553,0.542817,0.771409
8,2026-03-22 09:37:00,5116,1384,6500,3.70,3732,0.574154,0.787077
9,2026-03-22 09:38:00,2344,2155,4499,1.09,189,0.042009,0.521005


In [28]:
df["累计成交额"] = df["成交金额"].cumsum()
df["累计成交量"] = df["成交量"].cumsum()*100
df["VWAP"] = df["累计成交额"] / df["累计成交量"]

In [30]:
df.head(20)

,成交时间,成交价格,价格变动,成交量,成交金额,性质,分钟,买量,卖量,累计成交额,累计成交量,VWAP
0,2026-03-22 09:25:00,70.36,0.46,1315,9252340,买盘,2026-03-22 09:25:00,1315,0,9252340,131500,70.360000
1,2026-03-22 09:30:00,70.50,0.14,834,5871373,买盘,2026-03-22 09:30:00,834,0,15123713,214900,70.375584
2,2026-03-22 09:30:03,70.48,-0.02,374,2638263,卖盘,2026-03-22 09:30:00,0,374,17761976,252300,70.400222
3,2026-03-22 09:30:06,70.42,-0.06,652,4590780,卖盘,2026-03-22 09:30:00,0,652,22352756,317500,70.402381
4,2026-03-22 09:30:09,70.37,-0.05,264,1858127,卖盘,2026-03-22 09:30:00,0,264,24210883,343900,70.400939
5,2026-03-22 09:30:12,70.36,-0.01,518,3645070,卖盘,2026-03-22 09:30:00,0,518,27855953,395700,70.396646
6,2026-03-22 09:30:15,70.32,-0.04,185,1301441,卖盘,2026-03-22 09:30:00,0,185,29157394,414200,70.394481
7,2026-03-22 09:30:18,70.25,-0.07,80,562356,卖盘,2026-03-22 09:30:00,0,80,29719750,422200,70.392586
8,2026-03-22 09:30:21,70.13,-0.12,65,456086,卖盘,2026-03-22 09:30:00,0,65,30175836,428700,70.389167
9,2026-03-22 09:30:24,70.00,-0.13,151,1057270,卖盘,2026-03-22 09:30:00,0,151,31233106,443800,70.376534


In [31]:
df["VWAP斜率"] = df["VWAP"].diff()
df["均价抬高"] = df["VWAP斜率"] > 0

In [32]:
df.head(20)

,成交时间,成交价格,价格变动,成交量,成交金额,性质,分钟,买量,卖量,累计成交额,累计成交量,VWAP,VWAP斜率,均价抬高
0,2026-03-22 09:25:00,70.36,0.46,1315,9252340,买盘,2026-03-22 09:25:00,1315,0,9252340,131500,70.360000,NaN,False
1,2026-03-22 09:30:00,70.50,0.14,834,5871373,买盘,2026-03-22 09:30:00,834,0,15123713,214900,70.375584,0.015584,True
2,2026-03-22 09:30:03,70.48,-0.02,374,2638263,卖盘,2026-03-22 09:30:00,0,374,17761976,252300,70.400222,0.024638,True
3,2026-03-22 09:30:06,70.42,-0.06,652,4590780,卖盘,2026-03-22 09:30:00,0,652,22352756,317500,70.402381,0.002159,True
4,2026-03-22 09:30:09,70.37,-0.05,264,1858127,卖盘,2026-03-22 09:30:00,0,264,24210883,343900,70.400939,-0.001442,False
5,2026-03-22 09:30:12,70.36,-0.01,518,3645070,卖盘,2026-03-22 09:30:00,0,518,27855953,395700,70.396646,-0.004293,False
6,2026-03-22 09:30:15,70.32,-0.04,185,1301441,卖盘,2026-03-22 09:30:00,0,185,29157394,414200,70.394481,-0.002166,False
7,2026-03-22 09:30:18,70.25,-0.07,80,562356,卖盘,2026-03-22 09:30:00,0,80,29719750,422200,70.392586,-0.001894,False
8,2026-03-22 09:30:21,70.13,-0.12,65,456086,卖盘,2026-03-22 09:30:00,0,65,30175836,428700,70.389167,-0.003419,False
9,2026-03-22 09:30:24,70.00,-0.13,151,1057270,卖盘,2026-03-22 09:30:00,0,151,31233106,443800,70.376534,-0.012633,False
